In [4]:
from pathlib import Path

import pandas as pd
import pypsa

INPUT_DIR = Path("inputs") / "tamil_nadu_ra_2025_26"
n_year = pypsa.Network(INPUT_DIR)

# Select the Monday-Sunday week containing the annual demand peak.
annual_demand = n_year.loads_t.p_set.sum(axis=1)
peak_timestamp = annual_demand.idxmax()
week_start = peak_timestamp.normalize() - pd.Timedelta(days=peak_timestamp.weekday())
week_end = week_start + pd.Timedelta(days=7)
week = n_year.snapshots[(n_year.snapshots >= week_start) & (n_year.snapshots < week_end)]

assert len(week) == 168, f"Expected 168 hourly snapshots, found {len(week)}"
n = n_year.copy(snapshots=week)

print(f"Annual peak: {annual_demand.loc[peak_timestamp]:,.2f} MW at {peak_timestamp}")
print(f"Optimizing peak week: {week_start} to {week_end - pd.Timedelta(hours=1)}")

INFO:pypsa.network.io:Imported network 'Tamil Nadu FY 2025-26 single-node UC inputs' has buses, generators, loads, storage_units


Annual peak: 19,987.33 MW at 2025-07-11 16:00:00
Optimizing peak week: 2025-07-07 00:00:00 to 2025-07-13 23:00:00


In [5]:
n

PyPSA Network 'Tamil Nadu FY 2025-26 single-node UC inputs'
-----------------------------------------------------------
Components:
 - Bus: 1
 - Generator: 24
 - Load: 1
 - StorageUnit: 1
Snapshots: 168

In [ ]:
display(n.buses)
display(n.generators)
display(n.storage_units)

PyPSA Network 'Tamil Nadu FY 2025-26 single-node UC inputs'


,v_nom,type,x,y,carrier,unit,location,v_mag_pu_set,v_mag_pu_min,v_mag_pu_max,control,generator,sub_network
name,,,,,,,,,,,,,
Tamil_Nadu,230.0,,78.6569,11.1271,AC,,,1.0,0.0,inf,PQ,,


,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
nuclear_fleet,Tamil_Nadu,PQ,,1448.000000,0.0,False,0.0,inf,NaN,0.612008,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
hydro_reservoir,Tamil_Nadu,PQ,,1886.000000,0.0,False,0.0,inf,NaN,0.000000,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
wind_fleet,Tamil_Nadu,PQ,,9631.000000,0.0,False,0.0,inf,NaN,0.000000,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
solar_utility,Tamil_Nadu,PQ,,11106.000000,0.0,False,0.0,inf,NaN,0.000000,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
solar_DRE_rooftop,Tamil_Nadu,PQ,,2353.000000,0.0,False,0.0,inf,NaN,0.000000,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
STOA_MTOA_import,Tamil_Nadu,PQ,,6727.000000,0.0,False,0.0,inf,NaN,0.000000,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
unserved_energy,Tamil_Nadu,PQ,,100000.000000,0.0,False,0.0,inf,NaN,0.000000,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
coal_block_01,Tamil_Nadu,PQ,,1067.333333,0.0,False,0.0,inf,NaN,0.550000,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
coal_block_02,Tamil_Nadu,PQ,,1067.333333,0.0,False,0.0,inf,NaN,0.550000,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0


,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,state_of_charge_initial_per_period,state_of_charge_set,cyclic_state_of_charge,cyclic_state_of_charge_per_period,max_hours,efficiency_store,efficiency_dispatch,standing_loss,inflow,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
PSP_fleet,Tamil_Nadu,PQ,,400.0,0.0,False,0.0,inf,NaN,-0.95,...,False,NaN,True,False,6.0,0.894427,0.894427,0.0,0.0,0.0


DatetimeIndex(['2025-07-07 00:00:00', '2025-07-07 01:00:00',
               '2025-07-07 02:00:00', '2025-07-07 03:00:00',
               '2025-07-07 04:00:00', '2025-07-07 05:00:00',
               '2025-07-07 06:00:00', '2025-07-07 07:00:00',
               '2025-07-07 08:00:00', '2025-07-07 09:00:00',
               ...
               '2025-07-13 14:00:00', '2025-07-13 15:00:00',
               '2025-07-13 16:00:00', '2025-07-13 17:00:00',
               '2025-07-13 18:00:00', '2025-07-13 19:00:00',
               '2025-07-13 20:00:00', '2025-07-13 21:00:00',
               '2025-07-13 22:00:00', '2025-07-13 23:00:00'],
              dtype='datetime64[us]', name='snapshot', length=168, freq=None)

In [ ]:
status, termination_condition = n.optimize(
    solver_name="highs",
    formulation="kirchhoff",
)

if termination_condition != "optimal":
    raise RuntimeError(f"Optimization failed: {status}, {termination_condition}")

print(f"Optimization: {status} ({termination_condition})")

C:\Users\b076218\AppData\Local\Temp\ipykernel_42516\1245455147.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  status, termination_condition = n.optimize(
Index(['Tamil_Nadu'], dtype='object', name='name')
Index(['nuclear_fleet', 'hydro_reservoir', 'wind_fleet', 'solar_utility',
       'solar_DRE_rooftop', 'STOA_MTOA_import', 'unserved_energy',
       'coal_block_01', 'coal_block_02', 'coal_block_03', 'coal_block_05',
       'coal_block_04', 'coal_block_06', 'coal_block_07', 'coal_block_09',
       'coal_block_08', 'coal_block_10', 'coal_block_11', 'coal_block_12',
       'gas_block_01', 'gas_block_02', 'biomass_block_01', 'hydro_DRE',
       'biomass_block_02'],
      dtype='object', name='name')


       'coal_block_04', 'coal_block_06', 'coal_block_07', 'coal_block_09',
       'coal_block_08', 'coal_block_10', 'coal_block_11', 'coal_block_12',
       'gas_block_01', 'gas_block_02', 'biomass_block_01', 'biomass_block_02'],
      dtype='object', name='name').
Index(['Tamil_Nadu_demand'], dtype='object', name='name')
Index(['PSP_fleet'], dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - formulation: kirchhoff
INFO:linopy.io:Writing objective.
Writing binary variables.: 100%|██████████| 3/3 [00:00<00:00, 301.13it/s]
INFO:linopy.io: Writing time: 0.23s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 12600 primals, 33570 duals
Objective: 6.54e+09
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-status-p-fixed-upper, Generator-start_up-p-fixed-upper, Generator-shut_down-p-fixed-upper, Ge

Optimization: ok (optimal)


name,nuclear_fleet,hydro_reservoir,wind_fleet,solar_utility,solar_DRE_rooftop,STOA_MTOA_import,unserved_energy,coal_block_01,coal_block_02,coal_block_03,...,coal_block_09,coal_block_08,coal_block_10,coal_block_11,coal_block_12,gas_block_01,gas_block_02,biomass_block_01,hydro_DRE,biomass_block_02
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-07-07 00:00:00,886.187215,994.192392,2377.599373,-0.0,-0.0,4356.260683,-0.0,587.033333,587.033333,587.033333,...,587.033333,587.033333,587.033333,587.033333,587.033333,0.0,0.0,0.00,11.500337,0.000000
2025-07-07 01:00:00,886.187215,994.192392,2423.613398,-0.0,-0.0,-0.000000,-0.0,907.233333,907.233333,907.233333,...,907.233333,907.233333,907.233333,907.233333,907.233333,0.0,0.0,0.00,11.500337,0.000000
2025-07-07 02:00:00,886.187215,994.192392,2439.307925,-0.0,-0.0,-0.000000,-0.0,907.233333,907.233333,907.233333,...,907.233333,907.233333,907.233333,907.233333,654.895464,0.0,0.0,0.00,11.500337,0.000000
2025-07-07 03:00:00,886.187215,994.192392,2423.613398,-0.0,-0.0,-0.000000,-0.0,907.233333,907.233333,907.233333,...,907.233333,907.233333,907.233333,825.179991,587.033333,0.0,0.0,0.00,11.500337,0.000000
2025-07-07 04:00:00,886.187215,994.192392,2377.599373,-0.0,-0.0,-0.000000,-0.0,907.233333,907.233333,907.233333,...,907.233333,907.233333,907.233333,670.384017,587.033333,0.0,0.0,0.00,11.500337,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-13 19:00:00,886.187215,1104.806702,1918.868608,-0.0,-0.0,-0.000000,-0.0,907.233333,907.233333,907.233333,...,907.233333,907.233333,907.233333,907.233333,907.233333,0.0,0.0,237.75,12.779870,237.750000
2025-07-13 20:00:00,886.187215,994.192392,2038.080377,-0.0,-0.0,-0.000000,-0.0,907.233333,907.233333,907.233333,...,907.233333,907.233333,907.233333,907.233333,781.353012,0.0,0.0,237.75,11.500337,237.750000
2025-07-13 21:00:00,886.187215,994.192392,2157.292146,-0.0,-0.0,-0.000000,-0.0,907.233333,907.233333,907.233333,...,907.233333,907.233333,907.233333,907.233333,878.581244,0.0,0.0,237.75,11.500337,237.750000


In [ ]:
n.export_to_netcdf("tamil_nadu_2025_26.nc")
